## Step 2: Get Landsat Images from 2000-2020 for Classification

In [2]:
import ee
ee.Initialize(project="ee-joshisur231")
import geemap
import helpers.config as config
from helpers import landsat_composites 
import pandas as pd
import ast

In [3]:
noncrop_val = pd.read_csv(r"outputs\phenology_verified_for_validation\final_stable_nonCrop_manual_validation_samples.csv")
noncrop_val_remove = noncrop_val[noncrop_val["actual"] == 0]["point_id"].tolist()

crop_val = pd.read_csv(r"outputs\phenology_verified_for_validation\final_stable_ag_manual_validation_samples_glad.csv")
crop_val_remove = crop_val[crop_val["actual"] == 0]["point_id"].tolist()

In [16]:
stable_categories_nonCrop = [
    'STABLE NON-CROP (Rangeland)', 
    'STABLE NON-CROP (Forest)', 
    "STABLE NON-CROP (Barren/Water)"
]
noncrop_samples = pd.read_csv(r"outputs\phenology_verified_samples\stable_nonCrop_phenology_classification_results.csv")
noncrop_samples = noncrop_samples[noncrop_samples["status"].isin(stable_categories_nonCrop)]
noncrop_samples["stable_crop"] = 0
print(noncrop_samples.shape)
noncrop_samples = noncrop_samples[~noncrop_samples['point_id'].isin(noncrop_val_remove)]
print(noncrop_samples.shape)
noncrop_samples["coords"] = noncrop_samples["geo"].apply(lambda x: ast.literal_eval(x)["coordinates"])
noncrop_samples["x"] = noncrop_samples["coords"].apply(lambda x: x[0])
noncrop_samples["y"] = noncrop_samples["coords"].apply(lambda x: x[1])
noncrop_samples = noncrop_samples[["stable_crop", "x", "y"]]

crop_samples = pd.read_csv(r"outputs\phenology_verified_samples\stable_ag_phenology_classification_results_glad.csv")
crop_samples = crop_samples[crop_samples["status"] == 'STABLE CROPLAND']
crop_samples["stable_crop"] = 1
print(crop_samples.shape)
crop_samples = crop_samples[~crop_samples['point_id'].isin(crop_val_remove)]
print(crop_samples.shape)
crop_samples["coords"] = crop_samples["geo"].apply(lambda x: ast.literal_eval(x)["coordinates"])
crop_samples["x"] = crop_samples["coords"].apply(lambda x: x[0])
crop_samples["y"] = crop_samples["coords"].apply(lambda x: x[1])
crop_samples = crop_samples[["stable_crop", "x", "y"]]

(3673, 10)
(3669, 10)
(1898, 11)
(1875, 11)


In [17]:
samples = pd.concat([crop_samples, noncrop_samples])
samples = geemap.df_to_ee(samples, longitude="x", latitude="y")

In [20]:
geemap.ee_export_vector_to_asset(samples, description='crop_nonCrop_training_samples_glad', assetId='projects/ee-joshisur231/assets/agriculture_abandonment_nepal/crop_nonCrop_training_samples_glad')

projects/ee-joshisur231/assets/agriculture_abandonment_nepal/crop_nonCrop_training_samples_glad
Exporting crop_nonCrop_training_samples_glad... Please check the Task Manager from the JavaScript Code Editor.


In [21]:
start_date = "1999-01-01"
end_date = "2023-12-31"
crs = "EPSG:4326"
scale = 30

In [27]:
samples = ee.FeatureCollection('projects/ee-joshisur231/assets/agriculture_abandonment_nepal/crop_nonCrop_training_samples_glad')
processor = landsat_composites.C2SRExpressions()
unified_collection = processor._expression()\
    .map(lambda image: image.addBands(image.expression(processor.ALGORITHMS["NDVI"]).rename("ndvi")))\
    .map(lambda image: image.addBands(image.expression(processor.ALGORITHMS["EVI"]).rename("evi")))\
    .map(lambda image: image.addBands(image.expression(processor.ALGORITHMS["NDWI"]).rename("ndwi")))

In [28]:
combined_reducer = ee.Reducer.mean() \
    .combine(ee.Reducer.median(), sharedInputs=True) \
    .combine(ee.Reducer.stdDev(), sharedInputs=True) \
    .combine(ee.Reducer.percentile([25, 75]), sharedInputs=True)

In [29]:
def get_3yr_predictors(target_year):
    target_year = ee.Number(target_year)
    start_date = ee.Date.fromYMD(target_year.subtract(1), 1, 1)
    end_date = ee.Date.fromYMD(target_year.add(1), 12, 31)

    window_col = unified_collection.filterDate(start_date, end_date).filterBounds(config.ROI)

    predictors = window_col.reduce(combined_reducer)

    return predictors.set('year', target_year) \
                     .set('system:time_start', ee.Date.fromYMD(target_year, 1, 1).millis())

In [30]:
for year in range(2000, 2023):
    l_image = get_3yr_predictors(year)
    asset_id = f"projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples_glad/trainSamp_{str(int(year))}"
    
    training_data = l_image.reduceRegions(
        collection=samples,
        scale=scale,
        reducer=ee.Reducer.first(),
        tileScale = 12
    )
    geemap.ee_export_vector_to_asset(
        training_data, 
        description= f'export_train_{str(int(year))}', 
        assetId=asset_id
    )
    geemap.ee_export_vector_to_drive(
        training_data, 
        description= f'trainSamp_{str(int(year))}', 
        fileFormat = "csv",
        folder="aal"
    )
    # geemap.ee_export_image_to_asset(
    #     l_image, 
    #     description= f'export_train_image_{str(int(year))}', 
    #     assetId=asset_id,
    #     scale=scale,
    #     crs=crs,
    #     maxPixels=1e13,
    # )
print("Done!")

projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples_glad/trainSamp_2000
Exporting export_train_2000... Please check the Task Manager from the JavaScript Code Editor.
Exporting trainSamp_2000... Please check the Task Manager from the JavaScript Code Editor.
projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples_glad/trainSamp_2001
Exporting export_train_2001... Please check the Task Manager from the JavaScript Code Editor.
Exporting trainSamp_2001... Please check the Task Manager from the JavaScript Code Editor.
projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples_glad/trainSamp_2002
Exporting export_train_2002... Please check the Task Manager from the JavaScript Code Editor.
Exporting trainSamp_2002... Please check the Task Manager from the JavaScript Code Editor.
projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples_glad/trainSamp_2003
Exporting export_train_2003... Please check the Task 

### Validation

In [75]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
# from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
import ee
ee.Initialize(project="ee-joshisur231")
import geemap
Map = geemap.Map()

In [100]:
samples = ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples/train_2022")

In [101]:
samples = geemap.ee_to_df(samples, remove_geom=False).drop(columns=["geo"])

In [ ]:
predictors = ['blue_mean', 'blue_median', 'blue_p25', 'blue_p75', 'blue_stdDev',
       'green_mean', 'green_median', 'green_p25', 'green_p75', 'green_stdDev',
       'ndvi_mean', 'ndvi_median', 'ndvi_p25', 'ndvi_p75', 'ndvi_stdDev',
       'nir_mean', 'nir_median', 'nir_p25', 'nir_p75', 'nir_stdDev',
       'red_mean', 'red_median', 'red_p25', 'red_p75', 'red_stdDev',
       'swir1_mean', 'swir1_median', 'swir1_p25', 'swir1_p75',
       'swir1_stdDev', 'swir2_mean', 'swir2_median', 'swir2_p25', 'swir2_p75',
       'swir2_stdDev']
target = "stable_crop"
X = samples[predictors]
y = samples[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=45)

param_grid = {
    'n_estimators': [50, 100, 200, 300, 400, 500, 800], 
    'max_features': ['sqrt', 'log2'],
    # 'max_depth': [None, 10, 20, 30, 40, 50], 
    # 'min_samples_split': [2, 5, 10, 15], 
    'min_samples_leaf': [1, 2, 4, 6] 
}

rf_model = RandomForestClassifier(random_state=45)

kfold = KFold(n_splits=10, shuffle=True, random_state=45)

rf_search = RandomizedSearchCV(
    estimator=rf_model, 
    param_distributions=param_grid, 
    n_iter=50, 
    cv=kfold, 
    verbose=4, 
    random_state=45, 
    n_jobs=-1,
    scoring="f1"
)
rf_search.fit(X_train, y_train)

y_pred = rf_search.predict(X_test)

print(rf_search.best_params_)
print(rf_search.best_estimator_)
print(f"\nClassification Report")
print(classification_report(y_test, y_pred))

importance_df = pd.DataFrame({'Feature': predictors, 'Importance': rf_search.best_estimator_.feature_importances_})
importance_df["relative importance"] = importance_df["Importance"] * 100 / importance_df["Importance"].sum() 
print(importance_df.sort_values(by='Importance', ascending=False))
results_df = pd.DataFrame(rf_search.cv_results_)

columns_to_view = [
    'param_n_estimators', 
    # 'param_max_depth', 
    'mean_test_score',      
    'std_test_score',       
    'rank_test_score'       
]

best_models = results_df[columns_to_view].sort_values(by='rank_test_score').head(5)

print("\nBest Models")
print(best_models)

Fitting 10 folds for each of 50 candidates, totalling 500 fits
{'n_estimators': 100, 'min_samples_leaf': 2, 'max_features': 'log2'}
RandomForestClassifier(max_features='log2', min_samples_leaf=2, random_state=45)

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1113
           1       0.97      0.98      0.98       554

    accuracy                           0.98      1667
   macro avg       0.98      0.98      0.98      1667
weighted avg       0.98      0.98      0.98      1667

         Feature  Importance  relative importance
30    swir2_mean    0.111272            11.127223
5     green_mean    0.108811            10.881087
28     swir1_p75    0.076140             7.614008
6   green_median    0.074313             7.431284
20      red_mean    0.064917             6.491653
8      green_p75    0.061053             6.105281
33     swir2_p75    0.058805             5.880474
23       red_p75    0.052802         

KeyError: "['param_max_depth'] not in index"

## Making prediction

In [114]:
import ee
ee.Initialize(project="ee-joshisur231")
import geemap
import helpers.config as config
from helpers import landsat_composites 
import pandas as pd
import ast
import math

In [109]:
predictors = ['blue_mean', 'blue_median', 'blue_p25', 'blue_p75', 'blue_stdDev',
       'green_mean', 'green_median', 'green_p25', 'green_p75', 'green_stdDev',
       'ndvi_mean', 'ndvi_median', 'ndvi_p25', 'ndvi_p75', 'ndvi_stdDev',
       'nir_mean', 'nir_median', 'nir_p25', 'nir_p75', 'nir_stdDev',
       'red_mean', 'red_median', 'red_p25', 'red_p75', 'red_stdDev',
       'swir1_mean', 'swir1_median', 'swir1_p25', 'swir1_p75',
       'swir1_stdDev', 'swir2_mean', 'swir2_median', 'swir2_p25', 'swir2_p75',
       'swir2_stdDev']
samples_train = ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples/train_2022").filter(ee.Filter.notNull(predictors))
processor = landsat_composites.C2SRExpressions()
unified_collection = processor._expression()\
    .map(lambda image: image.addBands(image.expression(processor.ALGORITHMS["NDVI"]).rename("ndvi")))

In [110]:
combined_reducer = ee.Reducer.mean() \
    .combine(ee.Reducer.median(), sharedInputs=True) \
    .combine(ee.Reducer.stdDev(), sharedInputs=True) \
    .combine(ee.Reducer.percentile([25, 75]), sharedInputs=True)

In [111]:
def get_3yr_predictors(target_year):
    target_year = ee.Number(target_year)
    start_date = ee.Date.fromYMD(target_year.subtract(1), 1, 1)
    end_date = ee.Date.fromYMD(target_year.add(1), 12, 31)

    window_col = unified_collection.filterDate(start_date, end_date).filterBounds(config.ROI)

    predictors = window_col.reduce(combined_reducer)

    return predictors.set('year', target_year) \
                     .set('system:time_start', ee.Date.fromYMD(target_year, 1, 1).millis())

In [135]:
l_2022 = get_3yr_predictors(2022).clip(config.ROI)

In [136]:
# {'n_estimators': 100, 'min_samples_leaf': 2, 'max_features': 'log2'}
rf_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees= 100, #n_estimators
    minLeafPopulation = 2,#min_samples_leaf
    variablesPerSplit = round(math.log2(len(predictors))), #max_features
)\
    .setOutputMode('MULTIPROBABILITY')\
    .train(features = samples_train, classProperty="stable_crop", inputProperties=predictors)

classified_2022 = l_2022.classify(rf_classifier)
probabilities = classified_2022.arrayFlatten([["noncrop_prob", "crop_prob"]])

In [137]:
Map.addLayer(probabilities.select("crop_prob"), {"min":0, "max": 1, "palette":["red", "yellow", "green"]}, "prob")
Map

Map(bottom=14088.0, center=[27.409004644850572, 83.54501953045985], controls=(WidgetControl(options=['position…

In [54]:
def hyperparameter_tuning(samples, predictor_names, actual_class_col, split_frac, n_trees):
    samples = samples.randomColumn()
    train_samples = samples.filter(ee.Filter.lt("random", split_frac))
    val_samples = samples.filter(ee.Filter.gte("random", split_frac))

    def compute_acc(n_tree):
        n_tree = ee.Number(n_tree)
        classifier = ee.Classifier.smileRandomForest(n_tree)\
            .train(features=train_samples, classProperty=actual_class_col, inputProperties=predictor_names)

        error_mat = val_samples\
            .classify(classifier)\
            .errorMatrix(actual_class_col, "classification")
        
        acc = error_mat.accuracy()
        p_acc = error_mat.producersAccuracy()
        u_acc = error_mat.consumersAccuracy()
        f_score = error_mat.fscore()
        kappa = error_mat.kappa()

        acc_dict = {
            "error_matrix": error_mat.array(),
            "accuracy": acc,
            "kappa": kappa,
            "f": f_score,
            "producer": p_acc,
            "user": u_acc
         }

        return acc_dict
    
    accuracies = n_trees.map(compute_acc)
    return accuracies


In [58]:
hyperparameter_tuning(samples, predictors, "stable_crop", 0.7, ee.List([50, 150, 250, 350])).getInfo()

[{'accuracy': 0.9859241126070991,
  'error_matrix': [[1049, 10], [13, 562]],
  'f': [0.9891560584629892, 0.979947689625109],
  'kappa': 0.9691038623871463,
  'producer': [[0.9905571293673276], [0.9773913043478261]],
  'user': [[0.987758945386064, 0.9825174825174825]]},
 {'accuracy': 0.9859241126070991,
  'error_matrix': [[1049, 10], [13, 562]],
  'f': [0.9891560584629892, 0.979947689625109],
  'kappa': 0.9691038623871463,
  'producer': [[0.9905571293673276], [0.9773913043478261]],
  'user': [[0.987758945386064, 0.9825174825174825]]},
 {'accuracy': 0.9859241126070991,
  'error_matrix': [[1049, 10], [13, 562]],
  'f': [0.9891560584629892, 0.979947689625109],
  'kappa': 0.9691038623871463,
  'producer': [[0.9905571293673276], [0.9773913043478261]],
  'user': [[0.987758945386064, 0.9825174825174825]]},
 {'accuracy': 0.9865361077111383,
  'error_matrix': [[1050, 9], [13, 562]],
  'f': [0.9896324222431667, 0.9808027923211169],
  'kappa': 0.9704354090832081,
  'producer': [[0.9915014164305949